Optuna Hyperparameter Tuning for Quinone Transfer Learning

- Uses existing data pipeline (prepare_train_test_dataset + Molecule_data)
- Loads pre-trained models weights
- Tunes learning rate, weight decay, dropout, unfreeze depth, batch size
- Optimizes validation RMSE

In [1]:
import os
import json
import argparse
from typing import Tuple, Optional, Dict

import torch
import torch.nn as nn
import numpy as np
import pandas as pd

from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

import optuna

from torch_geometric.loader import DataLoader
import torch_geometric.transforms as T

from models.GRNNModel import RGNNPredictor
from Data_Prep.datacreator import prepare_train_test_dataset
from Data_Prep.Graph_Data import Molecule_data

In [2]:
# ------------------------------------------------------------
# Device Selection
# ------------------------------------------------------------
def select_device():

    seed = 43
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

        device_index = 7 if torch.cuda.device_count() > 7 else 0
        print(f"Using CUDA device cuda:{device_index}")

        return f"cuda:{device_index}"

    print("Using CPU")

    return "cpu"


In [3]:
# ------------------------------------------------------------
# Generic Molecular Dataset Loader
# ------------------------------------------------------------
def load_molecular_dataframe(csv_path, smiles_col, target_col):

    print("Loading dataset...")

    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Dataset not found: {csv_path}")

    df = pd.read_csv(csv_path)

    print("Raw shape:", df.shape)
    print("Columns:", list(df.columns))

    if smiles_col not in df.columns or target_col not in df.columns:
        raise ValueError(f"CSV must contain '{smiles_col}' and '{target_col}'")

    # Remove empty columns
    df = df.dropna(axis=1, how="all")

    # Remove rows with missing values
    df = df[df[smiles_col].notna() & df[target_col].notna()]

    # Remove empty SMILES
    df = df[df[smiles_col].astype(str).str.strip() != ""]

    # Convert target to numeric
    df[target_col] = pd.to_numeric(df[target_col], errors="coerce")

    df = df[df[target_col].notna()]

    # Rename columns for pipeline compatibility
    df = df.rename(columns={
        smiles_col: "SMILES_str",
        target_col: "target"
    })

    print("Cleaned shape:", df.shape)

    return df

In [4]:
# ------------------------------------------------------------
# Ensure Processed Graph Data Exists
# ------------------------------------------------------------
def ensure_processed_data(df, savepath):

    train_file = f"data/{savepath}processed/train_data_set.pt"
    test_file = f"data/{savepath}processed/test_data_set.pt"

    if os.path.isfile(train_file) and os.path.isfile(test_file):
        print("Processed dataset already exists.")
        return

    print("Creating graph dataset...")

    prepare_train_test_dataset(
        df,
        "SMILES_str",
        "target",
        savepath=savepath
    )

    print("Graph dataset created.")

In [5]:
# ------------------------------------------------------------
# Create DataLoaders
# ------------------------------------------------------------
def make_loaders(savepath, batch_size):

    transform = T.Compose([T.NormalizeFeatures(["x", "edge_attr"])])

    train_data = Molecule_data(
        root=f"data/{savepath}",
        dataset="train_data_set",
        y=None,
        smiles=None,
        transform=transform
    )

    test_data = Molecule_data(
        root=f"data/{savepath}",
        dataset="test_data_set",
        y=None,
        smiles=None,
        transform=transform
    )

    generator = torch.Generator().manual_seed(43)

    test_len = len(test_data)

    val_len = test_len // 2
    test_len = test_len - val_len

    val_data, final_test_data = torch.utils.data.random_split(
        test_data,
        [val_len, test_len],
        generator=generator
    )

    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=batch_size)
    test_loader = DataLoader(final_test_data, batch_size=batch_size)

    print(
        f"Dataset sizes → train:{len(train_data)} "
        f"val:{len(val_data)} "
        f"test:{len(final_test_data)}"
    )

    return train_loader, val_loader, test_loader

In [6]:
# ------------------------------------------------------------
# Load Pretrained Model
# ------------------------------------------------------------
def load_pretrained_model(device, dropout,model_path):

    model = RGNNPredictor(dropout=dropout).to(device)

    if not os.path.exists(model_path):
        print("Warning: pretrained model not found.")
        return None

    checkpoint = torch.load(model_path, map_location=device)

    model.load_state_dict(checkpoint)

    print("Loaded pretrained weights")

    return model


In [7]:
# ------------------------------------------------------------
# Progressive Unfreezing
# ------------------------------------------------------------
def apply_progressive_unfreezing(model, unfreeze_layers):

    for p in model.parameters():
        p.requires_grad = False

    total_layers = len(model.atom_convs)

    start = max(0, total_layers - unfreeze_layers)

    for i in range(start, total_layers):

        for p in model.atom_convs[i].parameters():
            p.requires_grad = True

        for p in model.atom_batchnorms[i].parameters():
            p.requires_grad = True

        for p in model.atom_grus[i].parameters():
            p.requires_grad = True

    for p in model.mol_conv.parameters():
        p.requires_grad = True

    for p in model.mol_gru.parameters():
        p.requires_grad = True

    for p in model.lin2.parameters():
        p.requires_grad = True

In [8]:
# ------------------------------------------------------------
# Training Loop
# ------------------------------------------------------------
def run_one_training(
    model,
    device,
    train_loader,
    val_loader,
    epochs,
    lr,
    weight_decay
):

    criterion = nn.MSELoss()

    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
        weight_decay=weight_decay
    )

    best_val = float("inf")

    for epoch in range(epochs):

        model.train()

        train_loss = 0

        for batch in train_loader:

            batch = batch.to(device)

            optimizer.zero_grad()

            out = model(batch.x, batch.edge_index, batch.batch)

            loss = criterion(out.view(-1), batch.y.view(-1))

            loss.backward()

            optimizer.step()

            train_loss += loss.item()

        model.eval()

        preds = []
        trues = []

        with torch.no_grad():

            for batch in val_loader:

                batch = batch.to(device)

                out = model(batch.x, batch.edge_index, batch.batch)

                preds.extend(out.cpu().numpy().flatten())

                trues.extend(batch.y.cpu().numpy().flatten())

        rmse = np.sqrt(mean_squared_error(trues, preds))

        if rmse < best_val:
            best_val = rmse

    return best_val

In [9]:
# ------------------------------------------------------------
# Optuna Objective
# ------------------------------------------------------------
def objective(trial, device, train_loader, val_loader, epochs):

    dropout = trial.suggest_float("dropout", 0.1, 0.6)

    unfreeze_layers = trial.suggest_int("unfreeze_layers", 1, 7)

    lr = trial.suggest_float("lr", 1e-5, 5e-3, log=True)

    weight_decay = trial.suggest_float("weight_decay", 1e-7, 1e-3, log=True)

    model_path = "saved_models/modelHomo.model"
    
    model = load_pretrained_model(device, dropout,model_path)

    apply_progressive_unfreezing(model, unfreeze_layers)

    rmse = run_one_training(
        model,
        device,
        train_loader,
        val_loader,
        epochs,
        lr,
        weight_decay
    )

    return rmse


In [11]:
# ------------------------------------------------------------
# Main
# ------------------------------------------------------------
CSV_PATH   = "quinone_molecules.csv"
SMILES_COL = "smiles"
TARGET_COL = "homo_eV"
TRIALS     = 60
EPOCHS     = 300
BATCH_SIZE = 16

device = select_device()
df = load_molecular_dataframe(CSV_PATH, SMILES_COL, TARGET_COL)

savepath = "processed_dataset/"
ensure_processed_data(df, savepath)

train_loader, val_loader, test_loader = make_loaders(savepath, BATCH_SIZE)

Using CUDA device cuda:7
Loading dataset...
Raw shape: (30, 7)
Columns: ['gdb_name', 'gdb_idx', 'qm9_dataset_pos', 'smiles', 'homo_eV', 'lumo_eV', 'bandgap_eV']
Cleaned shape: (30, 7)
Creating graph dataset...
Processed data not found at data/processed_dataset/processed/train_data_set.pt, building graphs...
Graph construction done. Saving to file.
Processed data not found at data/processed_dataset/processed/test_data_set.pt, building graphs...
Graph construction done. Saving to file.
Graph dataset created.
Dataset sizes → train:24 val:3 test:3


In [12]:
study = optuna.create_study(direction="minimize")
study.optimize(
    lambda trial: objective(trial, device, train_loader, val_loader, EPOCHS),
    n_trials=TRIALS
)

[I 2026-03-08 16:16:58,244] A new study created in memory with name: no-name-9c62fa89-60e9-4c9c-ae20-7444f3bba9d0


Loaded pretrained weights


[I 2026-03-08 16:17:29,874] Trial 0 finished with value: 0.0937246744079832 and parameters: {'dropout': 0.4374290234503537, 'unfreeze_layers': 5, 'lr': 0.0003002196251658315, 'weight_decay': 4.0724022191281585e-07}. Best is trial 0 with value: 0.0937246744079832.


Loaded pretrained weights


[I 2026-03-08 16:17:57,998] Trial 1 finished with value: 0.08372129566779091 and parameters: {'dropout': 0.3332532794983246, 'unfreeze_layers': 3, 'lr': 0.0023603642553396333, 'weight_decay': 1.2027588168122072e-07}. Best is trial 1 with value: 0.08372129566779091.


...

[I 2026-03-08 16:45:01,425] Trial 57 finished with value: 0.04179075650624246 and parameters: {'dropout': 0.5399289629674611, 'unfreeze_layers': 2, 'lr': 0.003926972578384172, 'weight_decay': 4.950747968690524e-07}. Best is trial 25 with value: 0.018801134008828435.


Loaded pretrained weights


[I 2026-03-08 16:45:28,027] Trial 58 finished with value: 0.05918321020583328 and parameters: {'dropout': 0.5432697045283572, 'unfreeze_layers': 1, 'lr': 0.0036573090784879445, 'weight_decay': 5.548758301025716e-07}. Best is trial 25 with value: 0.018801134008828435.


Loaded pretrained weights


[I 2026-03-08 16:45:56,393] Trial 59 finished with value: 0.05183479107026304 and parameters: {'dropout': 0.587808589555164, 'unfreeze_layers': 3, 'lr': 0.002720874486130969, 'weight_decay': 3.3027378090756156e-07}. Best is trial 25 with value: 0.018801134008828435.


In [13]:
print("Best parameters:")
print(study.best_params)

Best parameters:
{'dropout': 0.47781949009481517, 'unfreeze_layers': 7, 'lr': 0.0017709621539926227, 'weight_decay': 5.7858789749948225e-06}
